In [6]:
# ============================================================
# notebooks/07_contextual_features.py
# Run after 06_behavioral_profiling.py:
#   python notebooks/07_contextual_features.py
#
# PURPOSE: Compute 8 contextual deviation features per transaction
# by comparing each transaction against the account's cumulative
# history AT THAT POINT IN TIME. Also generates account-compromise
# labels for fusion model training.
#
# WHAT WAS FIXED (merged from fix_nb07_geodisp.py):
#   - geo_displacement: was using string comparison on float addr1 column
#     → replaced with float expanding-median baseline, abs diff / 500
#   - NaN guard: np.nan_to_num applied before saving any arrays
#   - Correlation table: wrapped with std check to handle NaN/zero-std cols
#   - Autoencoder: strict=False on load_state_dict (tolerates minor
#     architecture differences without crashing)
#
# OUTPUTS:
#   data/contextual_features.npy     — shape (N, 8), float32
#   data/fusion_labels.npy           — isFraud labels, float32
#   data/fusion_account_ids.npy      — card1 per row, int64
#   data/p1_scores_for_fusion.npy    — P1 risk scores (flat 0.5 if unavailable)
#   data/fusion_feature_names.json   — ordered feature name list
#   data/drift_norm_params.json      — {p5, p95} for autoencoder normalisation
#
# NOTE: p1_risk_score (feature 8) will be flat 0.5 until NB08 patches it.
# NB08 generates real scores and writes them back before training.
#
# RUNTIME: ~10–15 min
# ============================================================

In [21]:
# %% Requirements Cell: Run once to align your execution ecosystem
%pip install pandas pyarrow numpy joblib torch
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
# %% Cell 1: Imports
import os
import sys
import json
import time
import warnings
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn

ROOT      = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR  = os.path.join(ROOT, "data")
MODEL_DIR = os.path.join(ROOT, "models")
SNAP_PATH = os.path.join(DATA_DIR, "tx_snapshot.parquet")

sys.path.insert(0, ROOT)
print(f"[NB07] Root Workspace Directed to: {ROOT}")

[NB07] Root Workspace Directed to: f:\rxtj_phase_2


In [23]:
# %% Cell 2: Load snapshot
print("[NB07] Loading parquet snapshot vector...")
t0 = time.time()
tx = pd.read_parquet(SNAP_PATH)
print(f"  {len(tx):,} rows, {len(tx.columns)} columns successfully indexed in {time.time()-t0:.1f}s")
print(f"  Available Context Targets: {list(tx.columns)}")

[NB07] Loading parquet snapshot vector...
  590,540 rows, 15 columns successfully indexed in 0.0s
  Available Context Targets: ['TransactionID', 'card1', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'addr1', 'DeviceInfo', 'hour', 'amt_cumsum', 'amt_cumstd', 'txn_seq', 'velocity_1h', 'velocity_24h', 'device_novel']


In [24]:
# %% Cell 3: Load Phase 1 preprocessors
print("[NB07] Loading Phase 1 artifacts...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    imputer = joblib.load(os.path.join(MODEL_DIR, "imputer.pkl"))
    scaler  = joblib.load(os.path.join(MODEL_DIR, "scaler.pkl"))

# Fallback hotpatch in case minor version differences persist across environments
if not hasattr(imputer, "_fill_dtype"):
    imputer._fill_dtype = np.float32

RAW_DIM = int(imputer.n_features_in_)
print(f"  imputer orientation : {RAW_DIM} expected raw dimensions")
print(f"  scaler orientation  : {scaler.n_features_in_} scaled dimensions")

# ── Autoencoder Network Config Architecture (strict=False Tolerant Topology) ──
class _Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, 256),        nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, input_dim)
        )
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

ae_state    = torch.load(os.path.join(MODEL_DIR, "autoencoder.pt"), map_location="cpu")
ae_in_dim   = ae_state["encoder.0.weight"].shape[1]
autoencoder = _Autoencoder(ae_in_dim)
autoencoder.load_state_dict(ae_state, strict=False)
autoencoder.eval()
print(f"  autoencoder mapped  : input_dim={ae_in_dim} initialized (strict=False)")

[NB07] Loading Phase 1 artifacts...
  imputer orientation : 224 expected raw dimensions
  scaler orientation  : 224 scaled dimensions
  autoencoder mapped  : input_dim=224 initialized (strict=False)


In [25]:
# %% Cell 4: Feature 7 — behavioral drift (autoencoder reconstruction error)
print("[NB07] Computing behavioral drift metrics...")
pcd_map = {"W": 0, "H": 1, "C": 2, "S": 3, "R": 4}
X_raw   = np.full((len(tx), RAW_DIM), np.nan, dtype=np.float32)
X_raw[:, 0] = tx["TransactionAmt"].fillna(0).values.astype(np.float32)
X_raw[:, 1] = tx["hour"].values.astype(np.float32)
X_raw[:, 2] = tx["ProductCD"].map(pcd_map).fillna(-1).values.astype(np.float32)
X_raw[:, 3] = pd.to_numeric(tx["addr1"], errors="coerce").values.astype(np.float32)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    X_imp    = imputer.transform(X_raw)
    X_scaled = scaler.transform(X_imp).astype(np.float32)
print(f"  Imputed+scaled transformation dimensions: {X_scaled.shape}")

BATCH = 2048
recon_errors = []
with torch.no_grad():
    for s in range(0, len(X_scaled), BATCH):
        batch     = torch.FloatTensor(X_scaled[s:s+BATCH])
        recon, _  = autoencoder(batch)
        mse       = ((recon - batch) ** 2).mean(dim=1).cpu().numpy()
        recon_errors.extend(mse.tolist())
        if (s // BATCH) % 50 == 0:
            print(f"    Processed Batch Stream: {min(s+BATCH, len(X_scaled)):>7,}/{len(X_scaled):,}")

recon_errors = np.array(recon_errors, dtype=np.float32)
p5, p95      = np.percentile(recon_errors, 5), np.percentile(recon_errors, 95)
drift_scores = np.clip((recon_errors - p5) / (p95 - p5 + 1e-9), 0.0, 1.0)
print(f"  Recon evaluation bounds — min={recon_errors.min():.4f} mean={recon_errors.mean():.4f} max={recon_errors.max():.4f}")
print(f"  Normalized Drift Context boundaries — p5={p5:.4f} p95={p95:.4f}")

with open(os.path.join(DATA_DIR, "drift_norm_params.json"), "w") as f:
    json.dump({"p5": float(p5), "p95": float(p95)}, f)
print(f"  drift_norm_params.json saved cleanly to storage.")

[NB07] Computing behavioral drift metrics...
  Imputed+scaled transformation dimensions: (590540, 224)
    Processed Batch Stream:   2,048/590,540
    Processed Batch Stream: 104,448/590,540
    Processed Batch Stream: 206,848/590,540
    Processed Batch Stream: 309,248/590,540
    Processed Batch Stream: 411,648/590,540
    Processed Batch Stream: 514,048/590,540
  Recon evaluation bounds — min=0.1995 mean=0.2151 max=0.2530
  Normalized Drift Context boundaries — p5=0.2056 p95=0.2463
  drift_norm_params.json saved cleanly to storage.


In [26]:
# %% Cell 5: Feature 1 — amount_z_score
print("[NB07] Building baseline variance indicators...")
amt      = tx["TransactionAmt"].fillna(0).values.astype(np.float32)
cum_mean = tx["amt_cumsum"].fillna(0).values.astype(np.float32)
cum_std  = np.where(
    tx["amt_cumstd"].fillna(0).values.astype(np.float32) < 0.1,
    1.0,
    tx["amt_cumstd"].values.astype(np.float32)
)
f1_amount_z = np.clip((amt - cum_mean) / cum_std, -5.0, 5.0).astype(np.float32)
print(f"  F1 amount_z_score   — mean={f1_amount_z.mean():.3f} std={f1_amount_z.std():.3f}")

[NB07] Building baseline variance indicators...
  F1 amount_z_score   — mean=0.026 std=1.203


In [27]:
# %% Cell 6: Feature 2 — merchant_novelty
tx["pcd_int"] = tx["ProductCD"].map({"W":0,"H":1,"C":2,"S":3,"R":4}).fillna(-1).astype(int)

def _pcd_novelty(group):
    pcd, novel, counts = group["pcd_int"].values, np.ones(len(group), np.float32), {}
    for j, p in enumerate(pcd):
        if j > 0 and p in counts:
            novel[j] = 1.0 - (counts[p] / j)
        counts[p] = counts.get(p, 0) + 1
    return pd.Series(novel, index=group.index)

print("  Evaluating dynamic merchant product interaction traces...")
f2_merchant_novelty = (
    tx.groupby("card1", group_keys=False)
    .apply(_pcd_novelty)
    .values.astype(np.float32)
)
print(f"  F2 merchant_novelty — aggregated profile mean={f2_merchant_novelty.mean():.3f}")

  Evaluating dynamic merchant product interaction traces...
  F2 merchant_novelty — aggregated profile mean=0.231


In [28]:
# %% Cell 7: Feature 3 — geo_displacement (FIXED)
print("  Running float spatial drift mapping calculations...")
tx_snap          = tx.copy()
tx_snap["addr1_f"] = pd.to_numeric(tx_snap["addr1"], errors="coerce").fillna(0.0)
tx_snap["addr1_baseline"] = (
    tx_snap.groupby("card1")["addr1_f"]
    .transform(lambda s: s.expanding().median().shift(1))
    .fillna(tx_snap["addr1_f"])
)
raw_disp = np.abs(tx_snap["addr1_f"].values - tx_snap["addr1_baseline"].values) / 500.0
f3_geo_disp = np.clip(raw_disp, 0.0, 1.0).astype(np.float32)

nan_count   = np.isnan(f3_geo_disp).sum()
if nan_count:
    print(f"  [Correction] Neutralizing {nan_count} structural NaNs inside array fields.")
    f3_geo_disp = np.nan_to_num(f3_geo_disp, nan=0.0)
print(f"  F3 geo_displacement — mean={f3_geo_disp.mean():.3f} NaN={nan_count}")

  Running float spatial drift mapping calculations...
  F3 geo_displacement — mean=0.097 NaN=0


In [29]:
# %% Cell 8: Feature 4 — hour_deviation
def _hour_deviation(group):
    hours, dev, counts = group["hour"].values, np.zeros(len(group), np.float32), [0]*24
    for j, h in enumerate(hours):
        if j > 0:
            dev[j] = 1.0 - (counts[int(h)] / j)
        counts[int(h)] += 1
    return pd.Series(dev, index=group.index)

print("  Extrapolating temporal hourly behavioral adjustments...")
f4_hour_dev = (
    tx.groupby("card1", group_keys=False)
    .apply(_hour_deviation)
    .values.astype(np.float32)
)
print(f"  F4 hour_deviation   — mean={f4_hour_dev.mean():.3f}")

  Extrapolating temporal hourly behavioral adjustments...
  F4 hour_deviation   — mean=0.912


In [30]:
# %% Cell 9: Feature 5 — device_novelty (from NB06 snapshot)
f5_device_novel = tx["device_novel"].fillna(0.5).values.astype(np.float32)
print(f"  F5 device_novelty   — structural mean={f5_device_novel.mean():.3f}")

  F5 device_novelty   — structural mean=0.444


In [31]:
# %% Cell 10: Feature 6 — velocity_ratio
vel_1h   = tx["velocity_1h"].fillna(0).values.astype(np.float32)
vel_24h  = tx["velocity_24h"].fillna(0).values.astype(np.float32)
f6_vel_ratio = np.clip(vel_1h / (vel_24h / 24.0 + 1e-6), 0.0, 10.0).astype(np.float32)
print(f"  F6 velocity_ratio   — structural mean={f6_vel_ratio.mean():.3f}")

  F6 velocity_ratio   — structural mean=1.595


In [32]:
# %% Cell 11: Feature 7 — behavioral_drift (computed above)
f7_drift = drift_scores.astype(np.float32)
print(f"  F7 behavioral_drift — structural mean={f7_drift.mean():.3f}")

  F7 behavioral_drift — structural mean=0.236


In [33]:
# %% Cell 12: Feature 8 — p1_risk_score
p1_npy = os.path.join(DATA_DIR, "model_probs_full.npy")
if os.path.exists(p1_npy):
    f8_p1 = np.load(p1_npy).astype(np.float32).ravel()
    if len(f8_p1) != len(tx):
        print(f"  [Size Mismatch] model_probs_full.npy offset ({len(f8_p1)} ≠ {len(tx)}). Defaulting to 0.5 flat score arrays.")
        f8_p1 = np.full(len(tx), 0.5, np.float32)
    else:
        print(f"  F8 p1_risk_score — loaded from model_probs_full.npy mean={f8_p1.mean():.3f}")
else:
    p1_fallback = os.path.join(DATA_DIR, "model_probs.npy")
    if os.path.exists(p1_fallback):
        tmp = np.load(p1_fallback).astype(np.float32).ravel()
        if len(tmp) == len(tx):
            f8_p1 = tmp
            print(f"  F8 p1_risk_score — loaded via baseline model_probs.npy checkpoint mean={f8_p1.mean():.3f}")
        else:
            f8_p1 = np.full(len(tx), 0.5, np.float32)
            print(f"  F8 p1_risk_score — alignment footprint mismatched. Using neutral constant vector placeholder (0.5)")
    else:
        f8_p1 = np.full(len(tx), 0.5, np.float32)
        print(f"  F8 p1_risk_score — no structural files found. Initialized with flat uniform priority vectors (0.5)")

  F8 p1_risk_score — loaded from model_probs_full.npy mean=0.690


In [34]:
# %% Cell 13: Stack all 8 features
FEATURE_NAMES = [
    "amount_z_score", "merchant_novelty", "geo_displacement", "hour_deviation",
    "device_novelty", "velocity_ratio", "behavioral_drift", "p1_risk_score"
]

X_fusion = np.column_stack([
    f1_amount_z, f2_merchant_novelty, f3_geo_disp, f4_hour_dev,
    f5_device_novel, f6_vel_ratio, f7_drift, f8_p1
]).astype(np.float32)

print(f"\n[NB07] Compiled Fusion Matrix Profile Space: {X_fusion.shape}")
print("  Tracking Boundary Distribution Fields (min / mean / max / NaN):")
for i, name in enumerate(FEATURE_NAMES):
    col = X_fusion[:, i]
    nan_c = int(np.isnan(col).sum())
    print(f"    {name:<25}  {col.min():.3f} / {col.mean():.3f} / {col.max():.3f} / NaN={nan_c}")

if np.isnan(X_fusion).any():
    total_nan = int(np.isnan(X_fusion).sum())
    print(f"\n  [Patching Process] Replacing {total_nan} floating point NaNs with 0.0 prior to binary serialization.")
    X_fusion = np.nan_to_num(X_fusion, nan=0.0)


[NB07] Compiled Fusion Matrix Profile Space: (590540, 8)
  Tracking Boundary Distribution Fields (min / mean / max / NaN):
    amount_z_score             -5.000 / 0.026 / 5.000 / NaN=0
    merchant_novelty           0.000 / 0.231 / 1.000 / NaN=0
    geo_displacement           0.000 / 0.097 / 1.000 / NaN=0
    hour_deviation             0.000 / 0.912 / 1.000 / NaN=0
    device_novelty             0.000 / 0.444 / 1.000 / NaN=0
    velocity_ratio             0.000 / 1.595 / 10.000 / NaN=0
    behavioral_drift           0.000 / 0.236 / 1.000 / NaN=0
    p1_risk_score              0.676 / 0.690 / 0.715 / NaN=0


In [35]:
# %% Cell 14: Account-compromise labels
print("[NB07] Extracting verification classification masks...")
fraud_accounts = set(tx.loc[tx["isFraud"] == 1, "card1"].astype(str).unique())
print(f"  Compromised Unique Entity Handles Discovered: {len(fraud_accounts):,}")

y_fusion    = tx["isFraud"].fillna(0).values.astype(np.float32)
account_ids = tx["card1"].fillna(-1).values.astype(np.int64)

print(f"  Imbalance Ratio Matrix Target Profile: {int(y_fusion.sum()):,} / {len(y_fusion):,} ({100*y_fusion.mean():.2f}%)")

[NB07] Extracting verification classification masks...
  Compromised Unique Entity Handles Discovered: 1,740
  Imbalance Ratio Matrix Target Profile: 20,663 / 590,540 (3.50%)


In [36]:
# %% Cell 15: Save all outputs
print("[NB07] Archiving finalized target arrays...")
np.save(os.path.join(DATA_DIR, "contextual_features.npy"),  X_fusion)
np.save(os.path.join(DATA_DIR, "fusion_labels.npy"),         y_fusion)
np.save(os.path.join(DATA_DIR, "fusion_account_ids.npy"),    account_ids)
np.save(os.path.join(DATA_DIR, "p1_scores_for_fusion.npy"),  f8_p1)

with open(os.path.join(DATA_DIR, "fusion_feature_names.json"), "w") as f:
    json.dump(FEATURE_NAMES, f, indent=2)

print(f"  ✓ Saved contextual_features.npy  → {X_fusion.shape}")
print(f"  ✓ Saved fusion_labels.npy        → {y_fusion.shape}")
print(f"  ✓ Saved fusion_account_ids.npy   → {account_ids.shape}")
print(f"  ✓ Saved feature schema list references json configuration file.")

[NB07] Archiving finalized target arrays...
  ✓ Saved contextual_features.npy  → (590540, 8)
  ✓ Saved fusion_labels.npy        → (590540,)
  ✓ Saved fusion_account_ids.npy   → (590540,)
  ✓ Saved feature schema list references json configuration file.


In [37]:
# %% Cell 16: Feature–label correlations (NaN-safe with Zero-Std Checks)
print("[NB07] Feature–label correlations (positive value tracks predictive fraud indicators):")
for i, name in enumerate(FEATURE_NAMES):
    col = X_fusion[:, i]
    if np.std(col) > 0 and not np.isnan(col).any():
        corr = float(np.corrcoef(col, y_fusion)[0, 1])
        bar  = "█" * int(abs(corr) * 40)
        sign = "+" if corr >= 0 else "-"
    else:
        corr, bar, sign = 0.0, "", "?"
    print(f"  {name:<25} {sign}{abs(corr):.4f}  {bar}")

print(f"\n[NB07] ✓ Step execution sequence completely validated and verified.")
print(f"  Total Residual Outlier Vectors: {int(np.isnan(X_fusion).sum())}")
print(f"\n[NB07] PROCEED TO ENGINES → Initialize your model weights via notebooks/08_fusion_model_training.ipynb")

[NB07] Feature–label correlations (positive value tracks predictive fraud indicators):
  amount_z_score            +0.0254  █
  merchant_novelty          +0.0144  
  geo_displacement          -0.0088  
  hour_deviation            +0.0080  
  device_novelty            -0.0524  ██
  velocity_ratio            +0.0470  █
  behavioral_drift          +0.1517  ██████
  p1_risk_score             +0.1248  ████

[NB07] ✓ Step execution sequence completely validated and verified.
  Total Residual Outlier Vectors: 0

[NB07] PROCEED TO ENGINES → Initialize your model weights via notebooks/08_fusion_model_training.ipynb
